# 额外周末练习 — 第 2 周

## 练习目标（理念）

把第 1 周的技术问答器做成完整原型：加上 **Gradio UI**、**流式输出**、用 **system prompt** 提升专业性，并具备在模型间切换的能力。若能演示 **工具调用（tool use）** 还有加分。

更大胆的方向：加入音频输入/语音回复。可用 ChatGPT 或 Claude 辅助；有问题也可发邮件给讲师。

## 和第 2 周概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Gradio `ChatInterface` | 网页聊天 UI + examples |
| 流式 `stream=True` | `yield` 边生成边刷新 |
| System Prompt | 技术导师人设 + 何时调用日期工具 |
| Tool Use | `get_current_date`：问日期/时间时由本地函数返回 |

## 怎么跑

1. `.env` 里设置 `OPEN_ROUTER_API_KEY`
2. 从上到下运行；最后 `gr.ChatInterface(...).launch()` 打开界面
3. 试示例：「What is today's date?」应触发工具；其它题走普通流式讲解

商业联想：语言导师、公司入职助手、课程伴学 AI——同一套「UI + 流式 + 工具」骨架可复用。


In [10]:
# ========== 导入：环境、时间、OpenAI 兼容客户端、Gradio ==========

# 标准库 os：读环境变量里的 API Key
import os
# 标准库 json：本练习主要给工具生态预留（当前 handle 未强制用到）
import json
# datetime：工具 get_current_date 与回复页脚时间戳都要用
from datetime import datetime
# load_dotenv：把 .env 密钥读进进程环境，避免写死在代码里
from dotenv import load_dotenv
# OpenAI 客户端：这里实际指向 OpenRouter 的兼容端点
from openai import OpenAI
# Gradio：快速搭 ChatInterface 网页 UI
import gradio as gr


In [ ]:
# ========== 环境与 OpenRouter 客户端 ==========

# 加载 .env；override=True 用文件值覆盖已存在的同名环境变量
load_dotenv(override=True)
# 读取 OpenRouter 密钥（注意变量名是 OPEN_ROUTER_API_KEY）
api_key = os.getenv("OPEN_ROUTER_API_KEY")
# OpenRouter 的 OpenAI 兼容 API 根地址
OPEN_ROUTER_URL = "https://openrouter.ai/api/v1"

# 粗检：没有 key 或太短则提示设置；否则打印就绪（文案保持英文原样）
if not api_key or len(api_key) < 10:
    print("Set OPEN_ROUTER_API_KEY in .env")
else:
    print("API key Looks good.")

# 创建客户端：后续 chat.completions 都走 OpenRouter
openai = OpenAI(base_url=OPEN_ROUTER_URL, api_key=api_key)


API key Looks good.


In [12]:
# ========== 模型常量：经 OpenRouter 调用的 Claude ==========

# 模型 ID 字符串必须与 OpenRouter 目录一致，勿擅自改名
MODEL_CLAUDE = "anthropic/claude-3.5-haiku"


In [13]:
# ========== System Prompt：技术导师人设 + 何时用日期工具 ==========
# 整段 prompt 保留英文：改译会改变模型行为与工具触发条件

SYSTEM_PROMPT = """
You are a helpful technical tutor who answers questions about provided code or technical programming questions, software engineering, 
data science and LLMs. And other related topics to computer science.Give clear, structured explanations. Use markdown when useful (lists, code blocks). 
Do not wrap your whole reply in a code block.If the user asks for today's date or current time, use the get_current_date tool."""


In [14]:
# ========== 工具 Schema：get_current_date（无参数）==========
# 用 JSON Schema 描述函数，让模型知道「何时 / 如何」发起 tool call

get_current_date_spec = {
    "name": "get_current_date",
    "description": "Get the current date and time. Use when the user asks for today's date, current time, or what day it is.",
    "parameters": {
        "type": "object",
        # 无业务参数；additionalProperties=False 禁止模型乱塞字段
        "properties": {},
        "additionalProperties": False,
    },
}


In [15]:
# ========== 注册 tools 列表（OpenAI 风格 function calling）==========
# 传给 chat.completions.create(..., tools=tools)
tools = [{"type": "function", "function": get_current_date_spec}]


In [16]:
# ========== 工具处理：执行模型请求的 get_current_date ==========

def handle_tool_calls(message):
    # responses：将作为 role=tool 的消息追加回对话
    responses = []
    # 一轮可能有多个 tool_call；本练习主要处理日期工具
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_current_date":
            # 本地取当前时间并格式化成可读字符串（星期、月日年、时分）
            now = datetime.now().strftime("%A, %B %d, %Y. Time: %H:%M")
            responses.append({
                "role": "tool",
                "content": now,
                # tool_call_id 必须与模型请求里的 id 对应，否则 API 会报错
                "tool_call_id": tool_call.id,
            })
    return responses


In [21]:
# ========== 聊天主函数：工具循环 → 再流式生成最终回答 ==========

def chat(message, history):
    # Gradio history（messages 格式）→ 只保留 role/content
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    # 组装 API messages：system + 历史 + 当前 user
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        *history,
        {"role": "user", "content": message},
    ]

    # 若工具返回了日期，稍后展示在回复顶部
    date_display = ""
    # 第一轮：带 tools，让模型决定要不要调用 get_current_date
    response = openai.chat.completions.create(
        model=MODEL_CLAUDE,
        messages=messages,
        tools=tools,
    )
    # 若 finish_reason 是 tool_calls：执行工具、把结果塞回 messages，再请求，直到不再要工具
    while response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        tool_responses = handle_tool_calls(msg)
        # 记下工具返回的日期文本，供 UI 顶部展示
        for tr in tool_responses:
            if tr.get("content"):
                date_display = tr["content"]
                break
        # 先追加 assistant 的 tool 请求，再追加 tool 结果
        messages.append(msg)
        messages.extend(tool_responses)
        response = openai.chat.completions.create(
            model=MODEL_CLAUDE,
            messages=messages,
            tools=tools,
        )

    # 页脚时间戳：本轮开始流式前的本地时间
    reply_time = datetime.now()
    time_footer = reply_time.strftime("%d %b %Y, %H:%M")
    # 最终回答改为流式，便于 Gradio 边生成边刷新
    stream = openai.chat.completions.create(
        model=MODEL_CLAUDE,
        messages=messages,
        stream=True,
    )
    result = ""
    for chunk in stream:
        # 累计增量文本；delta.content 可能为 None
        result += chunk.choices[0].delta.content or ""
        # 有工具日期则加在正文前；每帧都带页脚时间
        body = (f"**Date & time:** {date_display}\n\n" + result) if date_display else result
        yield body + f"\n\n— {time_footer}"


In [ ]:
# ========== 启动 Gradio ChatInterface ==========

# fn=chat：用户每发一条就调用上面的生成器
# type="messages"：history 用 role/content 字典列表（与 OpenAI messages 对齐）
gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Technical Tutor ",
    description="Ask any technical question and get an answer.",
    # 点击示例可一键填入；含日期题用于演示 tool use
    examples=[
        "What does yield from do in Python?",
        "What is today's date?",
        "Explain attention in transformers in 3 sentences.",
    ],
    flagging_mode="never",
).launch()
